# 01 — Data Loading

Load all MIMIC-IV tables, infer schemas, and generate a dataset summary.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.loader import DataLoader
from src.utils.schema import schemas_to_dataframe
from src.utils.io_utils import save_parquet
from src.utils.config import CFG

loader = DataLoader()
tables = loader.load_all_small_tables()
summaries = {name: s for name, (_, s) in tables.items()}
schemas = {}
for name, (df, _) in tables.items():
    if not df.empty:
        from src.utils.schema import infer_table_schema
        t_cfg = CFG.tables.get(name, {})
        schemas[name] = infer_table_schema(df, name, t_cfg.get('date_cols'), t_cfg.get('cat_cols'), t_cfg.get('id_cols'))

summary_df = loader.generate_dataset_summary(summaries)
display(summary_df)
save_parquet(summary_df, CFG.resolve('reports/tables/notebook_load_summary.parquet'))